In [1]:
"""
PROJECT STEP 1: NOMINAL GROUND TRUTH EXTRACTION (STATIC PHASE)

DESCRIPTION:
This module establishes the core linearization anchor points ([A_nominal], [B_nominal]) 
for an open-loop unstable Continuous Stirred Tank Reactor (CSTR) modeled as a 
Multi-Input, Multi-Output (MIMO) dynamic system.

HARDWARE & EQUIPMENT SPECIFICATIONS:
- Primary Sensor Array (Outputs, y(t) or x(t)): Integrated dual-channel inline sensor 
  tracking a 2D continuous state vector x(t) = [C_A(t), T(t)]^T.
  * Channel 1 (Concentration): High-precision inline spectrophotometer.
    Operational Bounds: [0.0, 2.0] kmol/m^3. Sampling rate: 100 Hz (dt = 0.01s).
  * Channel 2 (Temperature): Industrial Class-A Resistance Temperature Detector (RTD).
    Operational Bounds: [300.0, 500.0] K. Sampling rate: 100 Hz (dt = 0.01s).
    
  Actuator Assembly (Control Input Vector, u(t)): Vector manipulating coolant temperature 
  which alters reactor temperature, which is coupled with reaction concentration due to rxn  
  rate. u(t) ∈ R^m simultaneously influences both concentration C_A(t) and reactor temperature  
  T(t) dynamics through the MIMO system matrix B_nominal.
  Physical Hardware Bounds: [250.0, 450.0] K (for thermal inputs / limits respectively).
================================================================================
"""

import numpy as np
from dataclasses import dataclass 
from scipy.linalg import solve_continuous_lyapunov

class Model_Initialization:
    """
    Class 1: Handles plant parameter storage, physical ODE evaluations, and simulates
    the real-time sensor streaming phase under input dither excitation. Only instantiated
    once to act as a parameter container (holding physical constants like k_0, E_R, etc.). 
    The class doesn't store the state variables as instance attributes.
    """
    def __init__(self):
        # 1. Hardware Sensor Setup
        self.dt = 0.01          # 100 Hz hardware sensor sampling frequency
        self.n_samples = 1000   # Number of continuous snapshots collected
               
        # 2. Plant Physical Ground-Truth Constants
        self.q_V = 1.0          # Volumetric space velocity (q/V) [s^-1]
        self.C_Af = 1.0         # Feed concentration of reactant A [kmol/m^3]
        self.T_f = 350.0        # Feed temperature [K]
        self.k_0 = 1e8          # Arrhenius pre-exponential kinetic constant [s^-1]
        self.E_R = 6000.0       # Activation energy over gas constant (E/R) [K]
        self.dH_term = 2e5      # Dimensionless adiabatic heat of reaction term [K*m^3/kmol]
        self.UA_term = 0.5      # Jacket heat transfer coefficient term [s^-1]

        # Pre-allocate state and control trajectories over time
        C_A = np.zeros(self.n_samples)
        T = np.zeros(self.n_samples)
        X_dot_storage = np.zeros((self.n_samples, 2)) # Stores vector field values at each time point in a matrix
        
        U_control = np.zeros(self.n_samples)          # Vector containing control inputs, which are the coolant's
                                                      # temp at various time points in K
    
        # 2. Set initial values (C_A0, T0)
        C_A[0] = self.C_Af
        T[0] = self.T_f
        U_control[0] = 350  # set equal to starting temp of reactor to make sure the system is in thermal
                            # equilibrium at runtime, giving controller time to adjust
    
    def cstr_nonlinear_dynamics(self, C_A, T, U_control):
        """
        Evaluates the exact, non-linear physical ordinary differential equations 
        (mass balance and energy balance) governing the internal material and energy 
        balances of the reactor. Takes in the current state vector in a loop and uses
        it to output the change in state variable values at that specific point.
        """
        # Arrhenius rate law expression
        reaction_rate = self.k_0 * np.exp(-self.E_R / T) * C_A
        
        # Mass Balance: d(C_A)/dt
        dC_A = self.q_V * (self.C_Af - C_A) - reaction_rate
        
        # Energy Balance: d(T)/dt
        dT = self.q_V * (self.T_f - T) + self.dH_term * reaction_rate - self.UA_term * (T - T_c)
        
        return np.array([dC_A, dT])

    def stream_sensor_data(self):
        """
        Responsible for simulating the open loop system. Calls upon cstr_nonlinear_dynamics
        to provide changes in state variables, which are used to inform the vector field
        variable x_dot.
        """
    
        for k in range(self.n_samples - 1):
        
            # Evaluate current time derivatives: x_dot = [dC_A/dt, dT/dt]
            X_dot = self.cstr_nonlinear_dynamics(C_A[k], T[k], U_control[k]) # Computes xdot at specific time point 
            X_dot_storage[k, :] = X_dot                                      # Stores derivative in matrix
        
            # Euler step: Vector update using derivatives
            C_A[k + 1] = C_A[k] + x_dot[0] * self.dt
            T[k + 1]   = T[k]   + x_dot[1] * self.dt
    
        # Construct of X 
        X_state = np.vstack([C_A, T])

        return X_state, U_control, X_dot_storage

class GetGroundTruth:
    """
    Class 2: Consumes the raw data streams from the Initialization pipeline,
    constructs the data-augmented matrix manifolds, and extracts the 
    nominal A and B matrices via linear least-squares regression.
    """
    def __init__(self, init_instance):
        self.plant = init_instance

    def compute_nominal_matrices(self):
        """
        Executes the data-augmented regression pipeline over the analytic 
        vector field snapshots to uncover the flat local linear grid.
        """
        # Fetch the active sensor and derivative streams from Class 1
        X, U, X_dot = self.plant.stream_sensor_data()
        
        # Data Augmentation: Stack State (X) and Control Input (U) into X_U_stack
        # Dimensions: (3 x N_SAMPLES)
        X_U_stack = np.vstack([X, U])
        
        # Execute Moore-Penrose Pseudoinverse to solve X_dot = [A | B] * X_U_stack
        A_B_stack = X_dot @ np.linalg.pinv(X_U_stack)
        
        # Slice the augmented mapping into independent nominal matrices
        A_nominal = A_B_stack[:, :2]
        B_nominal = A_B_stack[:, 2:3]
        
        return A_nominal, B_nominal


# ==============================================================================
# ROUTINE EXECUTION AND STABILITY VERIFICATION
# ==============================================================================
if __name__ == "__main__":
    # 1. Instantiate the classes
    plant_setup = Initialization()
    Ground_Truth_matrices = GetGroundTruth(plant_setup)
    
    # 2. Compute the exact ground-truth matrices
    A_nominal, B_nominal = Ground_Truth_matrices.compute_nominal_matrices()
    
    print("====================================================================")
    print("STATIC PHASE COMPLETE: NOMINAL GROUND TRUTH MATRICES EXTRACTION")
    print("====================================================================")
    print("A_nominal:\n", A_nominal)
    print("\nB_nominal:\n", B_nominal)
    
    # 3. Perform eigenvalue decomposition to prove open-loop positive stability
    eigenvalues = np.linalg.eigvals(A_nominal)
    print("\nOpen-Loop Eigenvalues (System Exponents):", eigenvalues)
    print("Verification: Contains positive exponent =", any(eigenvalues > 0))
    print("====================================================================")

NameError: name 'C_A' is not defined

In [ ]:
# """
# ================================================================================
# PROJECT STEP 2: LYAPUNOV STABILITY & EQUILIBRIUM-ELLIPSOID BARRIER SYNTHESIS
# ================================================================================
# EQUILIBRIUM-CENTRIC SAFE SET DESIGN DOCUMENTATION:
#
# Rather than assuming arbitrary operational factors, the safe set C is rigorously 
# derived from the equilibrium points and structural energy states of the system itself.
# Following Nagumo's theorem and framework, the safe set is defined as a permanent 
# geometric cage built directly around the local origin (equilibrium state x_e = 0):
#
#     C = { x in R^n : x^T * P * x <= delta^2 }
#
# The corresponding Control Barrier Function is explicitly modeled as:
#
#     h(x) = delta^2 - x^T * P * x
#
# Where:
# - P is the positive-definite matrix defining the ellipsoidal shape of the safe zone.
# - delta^2 is the maximum allowable potential energy deviation before the system 
#   ruptures its boundary cage.
#
# ARCHITECTURAL DESIGN DISCLAIMERS:
#
# 1. OPTIMIZATION SOLVER SELECTION:
#    We implement a quadratic Lyapunov function V(x) = x^T*P*x paired with a 
#    control-affine system. Because our performance bowl and operator safety 
#    boundaries result in constraints that remain strictly LINEAR with respect to 
#    the control input matrix [u], the online optimization reduces to a standard 
#    convex Quadratic Program (QP). We can therefore utilize standard, lightweight 
#    numerical solvers (such as SciPy's SLSQP). 
#    CRITICAL DISCLAIMER: If we were to implement more complex, non-quadratic, or 
#    non-smooth Lyapunov functions (e.g., Sum-of-Squares polynomials), the constraint 
#    space would become highly non-linear or semi-definite. This would require 
#    advanced, dedicated convex optimization parsing packages like CVXPY paired with 
#    high-performance solvers (e.g., OSQP, ECOS, or SCS).
#
# 2. HIGHER-ORDER LIE DERIVATIVES AND SYSTEM CHATTERING:
#    If the system has a control input was structurally decoupled from the safety   
#    states by multiple layers of integrators (Lg_h equals or approaches zero), we
#    would be forced to compute higher-order Lie derivatives (e.g., Lf^2_h, Lg_Lf_h) 
#    to find where the input appears. However, calculating higher-order derivatives
#    comes at the risk of amplifying noise in incoming sensor streams. This destabilizes
#    the QP solver's decision boundary, causing rapid, high-frequency switching of 
#    the control signal known as ACTUATOR CHATTERING, which rapidly degrades physical
#    valves and mechanical components.   
# ================================================================================
# """
#
# class ControlSynthesis:
#     """
#     Class 3: Consumes the ground-truth physical matrices (A, B) and nominal 
#     control laws to map the global Lyapunov stability and equilibrium-centered safety cages.
#     """
#     def __init__(self, A_nominal, B_nominal):
#         # Ground-Truth Physical State Space Vectors extracted from Step 1
#         self.A = A_nominal
#         self.B = B_nominal
#         
#         # SYNTHESIS GROUND TRUTH MATRIX (K):
#         # The gain matrix K is classified as a synthesis ground truth. While A and B 
#         # define the ground truth of the natural physics, K acts as the absolute mathematical 
#         # anchor point defining our nominal closed-loop performance intent. It is 
#         # mathematically impossible to construct the P ledger matrix without anchoring K first.
#         self.K = None          
#         self.P = None          # Lyapunov Tensor Shape Matrix (The Ledger of Tension)
#         self.delta_sq = None   # Maximum allowable energy boundary capacity (delta^2)
#
#     def define_lyapunov_function(self):
#         """
#         Executes the matrix operations to construct the Control Lyapunov Function bowl.
#         Formula: (A - B*K)^T * P + P * (A - B*K) = -Q
#         """
#         # Step 1: Establish the performance penalty ledger (Q)
#         Q = np.eye(2)
#         
#         # Step 2: Establish the Nominal Control Ground Truth (K)
#         # This matrix places the target closed-loop poles to stabilize the operating cage.
#         self.K = np.array([[4.0, 5.0]]) 
#         
#         # Step 3: Compute Closed-Loop State Transition Matrix
#         A_cl = self.A - self.B @ self.K
#         
#         # Step 4: Solve the Continuous-Time Algebraic Lyapunov Equation
#         # Generates the positive-definite matrix P defining the geometry of our bowl.
#         self.P = solve_continuous_lyapunov(A_cl.T, -Q)
#         
#         # Step 5: Derive Safe Set C Boundary dynamically from system eigenvalues
#         # We pull the open-loop eigenvalues and scale the allowable energy bound
#         # inversely proportional to the severity of the unstable thermal runaway branch.
#         eigenvalues = np.linalg.eigvals(self.A)
#         max_positive_exponent = np.max(eigenvalues[eigenvalues > 0])
#         
#         # delta^2: Maximum allowable deviation distance before system failure
#         self.delta_sq = 15.0 / max_positive_exponent 
#         
#         return self.P, self.K, self.delta_sq
#
#     def evaluate_lyapunov_stability(self, x, u_val):
#         """
#         Surveys the current coordinate to check how much the interference wave 
#         is compressing or expanding down the Lyapunov performance bowl.
#         Formula: V_dot = 2 * x^T * P * (A*x + B*u)
#         """
#         # Compute the continuous raw vector field matrix x_dot for this instant
#         x_dot = self.A @ x + self.B.flatten() * u_val
#         
#         # Calculate the scalar potential energy: V(x) = x^T * P * x
#         V = x.T @ self.P @ x
#         
#         # Calculate the temporal rate of change of the energy metric
#         V_dot = 2.0 * (x.T @ self.P @ x_dot)
#         
#         return V, V_dot
#
#     def evaluate_cbf(self, x):
#         """
#         Evaluates the safety barrier and explicitly computes the Lie Derivatives
#         to isolate natural system drift from actuator authority.
#         
#         Safety Manifold Definition (Nagumo Equilibrium Cage):
#         h(x) = delta_sq - x^T * P * x >= 0     
#         
#         ====================================================================
        #   BOUNDARY GEOMETRY STRESS TEST FOR dh/dx
        # ====================================================================
        # The local geometry of the gradient is probedd before coupling with f(x) and g(x).
        # Test: Apply a small virtual displacement vector (epsilon nudge) along the 
        # direction of the gradient and verify that the predicted change in h(x) 
        # matches linear expectation (Taylor expansion validation).
        # #epsilon = 1e-5

        # If dh_dx is a row vector, a nudge in the direction of dh_dx.T should increase h
        # #nudge_dir = dh_dx.T / (np.linalg.norm(dh_dx) + 1e-12)
        # #x_perturbed = x + epsilon * nudge_dir
         
        # # Re-evaluate h at the perturbed state
        # #h_perturbed = self.delta_sq - (x_perturbed.T @ self.P @ x_perturbed)
        # #h_linear_approximation = h + (dh_dx @ (x_perturbed - x))
        # 
        # Compute linearization residual error (should be close to zero for smooth quadratic bowls)
        # #gradient_sanity_error = np.abs(h_perturbed - h_linear_approximation)
        # 
        # if gradient_sanity_error > 1e-8:
        # #    print(f"[WARNING] Gradient geometry stress test flagged a high non-linearity residual: {gradient_sanity_error}")
        #  ====================================================================

#         Lie Derivative Formulations (SHOW WORK via Matrix Chain Rule):
#         dh_dx = -2 * x^T * P
#         Therefore:
#         - Lf_h = (dh_dx) * f(x) = -2 * x^T * P * (A * x)
#         - Lg_h = (dh_dx) * g(x) = -2 * x^T * P * B
#         """
#         # Step 1: Calculate the absolute scalar safety margin h(x)
#         h = self.delta_sq - (x.T @ self.P @ x)
#         
#         # Step 2: SHOW WORK - Compute the analytical gradient row matrix: dh/dx
#         # Size: Row Vector (1 x 2)
#         dh_dx = -2.0 * (x.T @ self.P)
#         
#         # Step 3: Isolate independent drift and control components of the vector field
#         # where x_dot = f(x) + g(x)u = Ax + Bu
#         f_x = self.A @ x
#         g_x = self.B
#         
#         # Step 4: SHOW WORK - Compute Analytical Lie Derivatives
#         # Lf_h tracks how the natural, unforced physical drift of the CSTR alters
#         # the system's position relative to the performance envelope edge.
#         # Matrix product chain: (1x2) @ (2x1) = scalar
#         Lf_h = dh_dx @ f_x
#         
#         # Lg_h maps the explicit authority your control valve possesses to squeeze
#         # or alter the instantaneous energy growth trajectory.
#         # Matrix product: (1x2) @ (2x1) = scalar
#         Lg_h = dh_dx @ g_x
#         
#         # Cast elements to floating-point scalars for the downstream QP loop
#         Lf_h_scalar = float(Lf_h)
#         Lg_h_scalar = float(Lg_h)
#         
#         return h, Lf_h_scalar, Lg_h_scalar


#def evaluate_cbf_robustness(dh_dx, g_x, sigma_x, f_x=None):
    """
    ================================================================================
    INDIVIDUAL LIE DERIVATIVE COMPONENT AUDITS & STRESS TESTING:
    
    1. DRIFT FLUX AUDIT (Testing Lf_h = (dh/dx) * f(x)):
       The drift vector field f(x) represents how the system moves naturally due to 
       gravity, inertia, or thermodynamics when actuators are turned off (u = 0).
       - Lf_h < 0: Nature is helping; physics naturally pulls the system away from danger.
       - Lf_h > 0: Nature is working against the system, accelerating toward a breach.
       
       OBSERVABILITY & STATE EVOLUTION PERSPECTIVE:
       Testing Lf_h acts as a structural test of system observability and 
       internal state propagation. By projecting the boundary gradient against the 
       unforced drift, we evaluate whether internal state trajectories are fully 
       observable through their natural energy evolution or if unobservable modes 
       are silently drifting toward the safety boundary without actuator visibility.

    2. ACTUATOR LEVERAGE AUDIT (Testing Lg_h = (dh/dx) * g(x)):
       The control matrix g(x) maps how actuators translate into state-space velocity.
       - Lg_h approx 0: Actuators are physically pointing parallel to the boundary wall.
       THE FATAL REALIZATION: At these states, actuators are completely useless for 
       safety. Even at maximum saturation, you cannot push the system away.
       If drift flux is heavily positive and actuator leverage is zero (Lg_h approx 0), 
       the system is mathematically uncontrollable at that coordinate, meaning no 
       controller can save it due to fundamental physical design flaws.
    ================================================================================

    Performs advanced Lie derivative component audits and stress testing:
    1. Drift Flux Audit ($L_f h$) if drift vector $f(x)$ is provided.
    2. Actuator Authority via Singular Value Decomposition (SVD) of $L_g h$.
    3. Boundary Variance Projection (Covariance Mapping) onto the safety boundary normal.
    
    Parameters:
    - dh_dx: Gradient of the barrier function w.r.t state, dh/dx (shape: (1, n) or (n,))
    - g_x: Control input matrix g(x) (shape: (n, m))
    - sigma_x: State estimation / disturbance covariance matrix \Sigma_x (shape: (n, n))
    - f_x: Optional unforced drift vector field f(x) (shape: (n,) or (n, 1))
    
    Returns:
    - audit_results: Dictionary containing Lf_h, SVD metrics, condition number, and projected variance.
    """
    # Ensure proper array shapes
#    dh_dx = np.atleast_2d(dh_dx) # Shape: (1, n)
    
    # Optional: Compute Drift Flux (Lf_h) if f_x is provided
#    Lf_h = None
#    Lf_h_scalar = 0.0
#    if f_x is not None:
#        f_x = np.atleast_2d(f_x)
#        if f_x.shape[0] == 1:  # if passed as row vector, make it column
#            f_x = f_x.T
#        Lf_h = dh_dx @ f_x
#        Lf_h_scalar = float(Lf_h.squeeze())

    # Compute Actuator Leverage Matrix: Lg_h = (dh/dx) * g(x)
    # Resulting shape: (1, m) where m is the number of actuators
#    Lg_h = dh_dx @ g_x
    
    # 1. Actuator Authority Audit via SVD
#    U, s_vals, Vt = np.linalg.svd(Lg_h)
#    sigma_min = s_vals[-1]
#    sigma_max = s_vals[0]
    
    # Compute Condition Number (\kappa)
    # Need to apply a threshold here later that checks condition number
    # and softens optimization constraints if it gets too high
#    eps = 1e-12
#    condition_number = sigma_max / (sigma_min + eps)
    
    # 2. Covariance Mapping to Boundary Normal
    # Calculates the exact scalar variance of the safety margin
#    sigma_h_squared = dh_dx @ sigma_x @ dh_dx.T
#    sigma_h_squared_scalar = float(sigma_h_squared.squeeze())
    
#    audit_results = {
#        "Lf_h": Lf_h_scalar if f_x is not None else None,
#        "Lg_h": Lg_h,
#        "sigma_min": float(sigma_min),
#        "sigma_max": float(sigma_max),
#        "condition_number": float(condition_number),
#        "safety_margin_variance": sigma_h_squared_scalar,
#        "is_paralyzed": sigma_min < 1e-3,
#        "is_ill_conditioned": condition_number > 100.0,
#        "nature_is_helping": Lf_h_scalar < 0.0 if f_x is not None else None
#    }
    
#    return audit_results
